# 1. 安装依赖并导入库

In [3]:
# 安装所需库（首次运行时执行）
!pip install jieba nltk

# 导入库
import re
import math
import jieba
import threading
from queue import Queue
from collections import defaultdict, Counter
import nltk
from nltk.probability import FreqDist
from nltk.text import Text

# 下载NLTK资源（首次运行需下载）
# nltk.download('punkt')
# nltk.download('stopwords')

# 2. 定义数据结构与工具类

In [9]:
class WordInfo:
    def __init__(self, word):
        self.word = word
        self.freq = 0
        self.left = Counter()  # 左侧字符/词
        self.right = Counter() # 右侧字符/词
        self.mi = 0.0          # 互信息
        self.left_entropy = 0.0
        self.right_entropy = 0.0
        self.lang = 'zh' if regex.search(r'\p{Han}', word) else 'en'  # 标记语言类型（中/英）

    def __repr__(self):
        return (f"WordInfo(word='{self.word}', lang={self.lang}, freq={self.freq}, "
                f"mi={self.mi:.2f}, left_entropy={self.left_entropy:.2f}, "
                f"right_entropy={self.right_entropy:.2f})")


class WordExtractor:
    def __init__(self, 
                 min_len_zh=2, max_len_zh=5,  # 中文词长度范围
                 min_len_en=2, max_len_en=10, # 英文词长度范围
                 mi_threshold_zh=2.5, mi_threshold_en=2.0,  # 互信息阈值（英文宽松些）
                 entropy_threshold_zh=1.0, entropy_threshold_en=0.8):  # 熵阈值
        # 中文参数
        self.min_len_zh = min_len_zh
        self.max_len_zh = max_len_zh
        self.mi_threshold_zh = mi_threshold_zh
        self.entropy_threshold_zh = entropy_threshold_zh
        # 英文参数
        self.min_len_en = min_len_en
        self.max_len_en = max_len_en
        self.mi_threshold_en = mi_threshold_en
        self.entropy_threshold_en = entropy_threshold_en
        
        self.word_dict = defaultdict(lambda: WordInfo(""))
        self.char_freq_zh = Counter()  # 中文单字频次
        self.char_freq_en = Counter()  # 英文单字符频次
        self.total_chars_zh = 0
        self.total_chars_en = 0
        self.lock = threading.Lock()

    def _is_zh_word(self, candidate):
        """判断候选词是否为中文词（含至少一个汉字）"""
        return regex.search(r'\p{Han}', candidate) is not None

    def _is_en_word(self, candidate):
        """判断候选词是否为英文词（仅含字母、数字、连字符）"""
        return regex.fullmatch(r'[A-Za-z0-9\-]+', candidate) is not None

    def _process_chunk(self, chunk):
        """处理文本块，区分中英文候选词"""
        local_word_dict = defaultdict(lambda: WordInfo(""))
        local_char_zh = Counter()
        local_char_en = Counter()

        # 预处理：保留中英文、数字及基本标点
        chunk = regex.sub(r'[^\p{Han}\p{Latin}0-9\s，。,.:;()\-]', '', chunk)
        chunk = re.sub(r'\s+', ' ', chunk).strip()
        if not chunk:
            return local_word_dict, local_char_zh, local_char_en

        # 分离中英文字符并统计
        chars = list(chunk)
        for c in chars:
            if regex.search(r'\p{Han}', c):
                local_char_zh[c] += 1
            elif regex.search(r'[A-Za-z]', c):
                local_char_en[c] += 1

        # 1. 中文候选词生成（滑动窗口+jieba辅助）
        jieba_words = jieba.lcut(chunk)  # jieba处理中文分词
        for i in range(len(chars)):
            # 根据中文长度范围生成候选词
            max_len = self.max_len_zh if self._is_zh_word(chars[i]) else self.max_len_en
            min_len = self.min_len_zh if self._is_zh_word(chars[i]) else self.min_len_en
            
            for l in range(min_len, max_len + 1):
                end = i + l
                if end > len(chars):
                    break
                candidate = ''.join(chars[i:end])
                
                # 中文过滤：排除jieba已识别的短词，避免重复
                if self._is_zh_word(candidate) and candidate in jieba_words and len(candidate) < 2:
                    continue
                # 英文过滤：排除单字母和无意义组合
                if self._is_en_word(candidate) and len(candidate) < self.min_len_en:
                    continue

                # 统计候选词信息
                info = local_word_dict[candidate]
                info.word = candidate
                info.freq += 1
                # 左侧字符（越界记为<start>）
                left_char = chars[i-1] if i > 0 else '<start>'
                info.left[left_char] += 1
                # 右侧字符（越界记为<end>）
                right_char = chars[end] if end < len(chars) else '<end>'
                info.right[right_char] += 1

        return local_word_dict, local_char_zh, local_char_en

    def _merge_results(self, local_dicts, local_chars_zh, local_chars_en):
        """合并多线程结果，区分中英文统计"""
        self.char_freq_zh = sum(local_chars_zh, Counter())
        self.char_freq_en = sum(local_chars_en, Counter())
        self.total_chars_zh = sum(self.char_freq_zh.values())
        self.total_chars_en = sum(self.char_freq_en.values())

        for local_dict in local_dicts:
            for word, info in local_dict.items():
                global_info = self.word_dict[word]
                global_info.word = word
                global_info.freq += info.freq
                global_info.left.update(info.left)
                global_info.right.update(info.right)
                global_info.lang = info.lang  # 同步语言标记

    def _calculate_metrics(self):
        """计算互信息和熵，区分中英文阈值"""
        for word, info in self.word_dict.items():
            if info.freq < 3:  # 降低低频阈值，适应英文可能的低频专业词
                continue

            # 1. 互信息计算（区分中英文字符集）
            min_mi = float('inf')
            char_freq = self.char_freq_zh if info.lang == 'zh' else self.char_freq_en
            total_chars = self.total_chars_zh if info.lang == 'zh' else self.total_chars_en

            for i in range(1, len(word)):
                left_part = word[:i]
                right_part = word[i:]
                # 英文允许子词含连字符（如"GPT-4"拆分为"GPT-"和"4"）
                p_left = char_freq.get(left_part, 1) / total_chars if total_chars > 0 else 0
                p_right = char_freq.get(right_part, 1) / total_chars if total_chars > 0 else 0
                p_word = info.freq / total_chars if total_chars > 0 else 0

                if p_left * p_right == 0:
                    mi = 0.0
                else:
                    mi = math.log2(p_word / (p_left * p_right))
                min_mi = min(min_mi, mi)
            info.mi = min_mi

            # 2. 信息熵计算
            left_total = sum(info.left.values())
            info.left_entropy = -sum(
                (c / left_total) * math.log2(c / left_total) 
                for c in info.left.values() if c > 0
            ) if left_total > 0 else 0.0

            right_total = sum(info.right.values())
            info.right_entropy = -sum(
                (c / right_total) * math.log2(c / right_total) 
                for c in info.right.values() if c > 0
            ) if right_total > 0 else 0.0

    def extract(self, text, top_n=100, num_threads=4):
        """抽取新词（支持中英文混合文本）"""
        chunks = [text[i:i + len(text)//num_threads] for i in range(0, len(text), len(text)//num_threads)]

        q = Queue()
        local_dicts = []
        local_chars_zh = []
        local_chars_en = []

        def worker():
            while True:
                chunk = q.get()
                if chunk is None:
                    break
                local_dict, zh_char, en_char = self._process_chunk(chunk)
                with self.lock:
                    local_dicts.append(local_dict)
                    local_chars_zh.append(zh_char)
                    local_chars_en.append(en_char)
                q.task_done()

        for _ in range(num_threads):
            threading.Thread(target=worker, daemon=True).start()
        for chunk in chunks:
            q.put(chunk)
        q.join()
        for _ in range(num_threads):
            q.put(None)

        # 合并结果并计算指标
        self._merge_results(local_dicts, local_chars_zh, local_chars_en)
        self._calculate_metrics()

        # 筛选：区分中英文阈值
        filtered = []
        for info in self.word_dict.values():
            if info.freq < 3:
                continue
            # 中文词阈值
            if info.lang == 'zh':
                if (info.mi >= self.mi_threshold_zh and
                    info.left_entropy >= self.entropy_threshold_zh and
                    info.right_entropy >= self.entropy_threshold_zh):
                    filtered.append(info)
            # 英文词阈值
            else:
                if (info.mi >= self.mi_threshold_en and
                    info.left_entropy >= self.entropy_threshold_en and
                    info.right_entropy >= self.entropy_threshold_en):
                    filtered.append(info)

        # 按频次排序
        filtered.sort(key=lambda x: x.freq, reverse=True)
        return filtered[:top_n]

3. 整合 NLTK 文本处理与新词抽取的主流程

In [11]:
# 中英文混合示例文本
sample_text = """
自然语言处理（NLP）是人工智能的重要分支。近年来，大语言模型（如GPT-4、LLaMA）推动了NLP技术的快速发展。
用户对智能客服、machine translation、情感分析等应用的需求日益增长。在电商领域，用户评论的情感分析可帮助企业优化产品。
此外，新词抽取技术能从海量文本中发现新兴词汇，如"元宇宙"、"AI生成内容"、"AIGC"等，对舆情监控至关重要。
大语言模型的出现，让natural language understanding和生成能力大幅提升，但也面临数据隐私、bias等挑战。
未来，NLP将与多模态技术结合，实现更智能的human-computer interaction。
"""

# --------------------------
# 步骤1：NLTK中英文文本处理
# --------------------------
print("===== NLTK中英文文本处理 =====")

# 1.1 分词（中文用jieba，英文用nltk）
# 中文分词
zh_seg = jieba.lcut(sample_text)
# 英文分词（先提取英文片段再用nltk）
en_pattern = regex.compile(r'[A-Za-z0-9\-]+')
en_tokens = en_pattern.findall(sample_text)
en_seg = [token.lower() for token in en_tokens]  # 英文小写化

# 合并分词结果（保留原始顺序）
all_tokens = []
i = 0
while i < len(sample_text):
    # 匹配中文
    if regex.match(r'\p{Han}', sample_text[i]):
        # 找到下一个非中文字符
        j = i
        while j < len(sample_text) and regex.match(r'\p{Han}', sample_text[j]):
            j += 1
        all_tokens.extend(jieba.lcut(sample_text[i:j]))
        i = j
    # 匹配英文
    elif regex.match(r'[A-Za-z0-9\-]', sample_text[i]):
        j = i
        while j < len(sample_text) and regex.match(r'[A-Za-z0-9\-]', sample_text[j]):
            j += 1
        all_tokens.append(sample_text[i:j].lower())
        i = j
    else:
        i += 1

k = 50
print(f"混合分词结果（前{k}个）：{all_tokens[:k]}")

# 1.2 去除停用词（中英文分别处理）
# 中文停用词
zh_stopwords = {'的', '是', '（', '）', '如', '等', '在', '此外', '让', '但', '也', '了', '可'}
# 英文停用词（用nltk）
en_stopwords = set(nltk.corpus.stopwords.words('english'))
# 过滤
filtered_tokens = [
    token for token in all_tokens
    if (token not in zh_stopwords and token not in en_stopwords)
    and len(token) > 1  # 过滤单字符
]
print(f"去停用词后（前15个）：{filtered_tokens[:15]}")

# 1.3 NLTK词频统计（区分中英文）
fdist = FreqDist(filtered_tokens)
print("\n高频词统计（前5个）：")
for word, count in fdist.most_common(5):
    lang = '中文' if regex.search(r'\p{Han}', word) else '英文'
    print(f"{word}（{lang}）: {count}次")

# 1.4 上下文分析（中英文分别示例）
nltk_text = Text(filtered_tokens)
print("\n'NLP'的上下文：")
nltk_text.concordance('nlp')  # 英文词示例
print("\n'自然语言'的上下文：")
nltk_text.concordance('自然语言')  # 中文词示例


# --------------------------
# 步骤2：中英文新词抽取
# --------------------------
print("\n===== 中英文新词抽取结果 =====")

extractor = WordExtractor(
    # 中文参数
    min_len_zh=2, max_len_zh=6,
    mi_threshold_zh=2.0, entropy_threshold_zh=0.8,
    # 英文参数（更宽松）
    min_len_en=2, max_len_en=10,
    mi_threshold_en=1.5, entropy_threshold_en=0.5
)

new_words = extractor.extract(sample_text, top_n=15, num_threads=2)

for idx, word_info in enumerate(new_words, 1):
    print(f"{idx}. {word_info}")

===== NLTK中英文文本处理 =====
混合分词结果（前50个）：['自然语言', '处理', 'nlp', '是', '人工智能', '的', '重要', '分支', '近年来', '大', '语言', '模型', '如', 'gpt-4', 'llama', '推动', '了', 'nlp', '技术', '的', '快速', '发展', '用户', '对', '智能', '客服', 'machine', 'translation', '情感', '分析', '等', '应用', '的', '需求', '日益增长', '在', '电商', '领域', '用户', '评论', '的', '情感', '分析', '可', '帮助', '企业', '优化', '产品', '此外', '新词']
去停用词后（前15个）：['自然语言', '处理', 'nlp', '人工智能', '重要', '分支', '近年来', '语言', '模型', 'gpt-4', 'llama', '推动', 'nlp', '技术', '快速']

高频词统计（前5个）：
nlp（英文）: 3次
技术（中文）: 3次
语言（中文）: 2次
模型（中文）: 2次
用户（中文）: 2次

'NLP'的上下文：
Displaying 3 of 3 matches:
自然语言 处理 nlp 人工智能 重要 分支 近年来 语言 模型 gpt-4 llama 推动 n
p 人工智能 重要 分支 近年来 语言 模型 gpt-4 llama 推动 nlp 技术 快速 发展 用户 智能 客服 machine translation
nding 生成 能力 大幅 提升 面临 数据 隐私 bias 挑战 未来 nlp 模态 技术 结合 实现 智能 human-computer interac

'自然语言'的上下文：
Displaying 1 of 1 matches:
 自然语言 处理 nlp 人工智能 重要 分支 近年来 语言 模型 gpt-4 ll

===== 中英文新词抽取结果 =====
1. WordInfo(word='。 ', lang=en, freq=4, mi=8.60, left_entropy=2.00, right_entropy=2.00)
2. WordInfo(word=

In [8]:
import regex  # 更强大的正则库，支持Unicode属性
word='nlp'
lang = 'zh' if regex.search(r'\p{Han}', word) else 'en' 
print(f'{word} belongs to {lang}.')

word='自然语言'
lang = 'zh' if regex.search(r'\p{Han}', word) else 'en' 
print(f'{word} belongs to {lang}.')


nlp belongs to en.
自然语言 belongs to zh.
